# 03 · Segmentacion exploratoria de clientes

Ruta de continuacion con las seis fases CRISP-DM. Leemos los CSV publicos del repositorio.

## 1. Comprension del negocio

Buscamos patrones de compra similares que merezcan investigacion. Los grupos son descriptivos, no categorias verdaderas, perfiles fijos ni decisiones sobre personas.

## 2. Comprension de los datos y EDA

Asociamos pedidos entregados, clientes seudonimos, lineas con precio y categorias. Primero exploramos estados, fechas y precios; despues la distribucion de agregados por cliente. Olist no tiene inventario ni cantidad explicita por linea.


In [ ]:
from urllib.error import HTTPError, URLError

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.base import clone
from sklearn.cluster import Birch, KMeans
from sklearn.decomposition import PCA
from sklearn.mixture import GaussianMixture
from sklearn.metrics import davies_bouldin_score, silhouette_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler

DATA_BASE_URL = "https://raw.githubusercontent.com/JuanParias29/ML_MiniWorkshop/main/data"
def load_dataset_csv(filename):
    url = f"{DATA_BASE_URL}/{filename}"
    try:
        return pd.read_csv(url)
    except (HTTPError, URLError) as error:
        raise RuntimeError(f"No se pudo cargar {filename} desde {url}. Compruebe la conexion a internet.") from error
orders = load_dataset_csv("olist_orders_dataset.csv")
customers = load_dataset_csv("olist_customers_dataset.csv")
items = load_dataset_csv("olist_order_items_dataset.csv")
products = load_dataset_csv("olist_products_dataset.csv")
print(f"Pedidos: {len(orders):,}; filas de articulos: {len(items):,}")


### EDA de tablas fuente

Inspeccionamos volumen por estado, compras por mes y precio por linea. Si la distribucion de precio tiene cola derecha, una transformacion logaritmica evita que algunos valores grandes dominen por escala; no implica que sean registros incorrectos.


In [ ]:
orders["order_purchase_timestamp"] = pd.to_datetime(orders["order_purchase_timestamp"], errors="coerce")
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
orders["order_status"].value_counts().plot.bar(ax=axes[0], color="slateblue")
axes[0].set_title("Pedidos por estado")
monthly_orders = orders.dropna(subset=["order_purchase_timestamp"]).set_index("order_purchase_timestamp").resample("MS").size()
monthly_orders.plot(ax=axes[1], color="darkorange")
axes[1].set_title("Pedidos por mes")
axes[1].set_ylabel("Pedidos")
np.log1p(items["price"].clip(lower=0)).plot.hist(bins=35, ax=axes[2], color="teal")
axes[2].set_title("Precio por línea — log(1+x)")
axes[2].set_xlabel("Precio transformado")
plt.tight_layout()
plt.show()
print(f"Cobertura de compra: {orders['order_purchase_timestamp'].min().date()} a {orders['order_purchase_timestamp'].max().date()}")
print(f"Precio BRL: mediana={items['price'].median():.2f}; media={items['price'].mean():.2f}; sesgo={items['price'].skew():.2f}")


## 3. Preparacion: variables por cliente y justificacion

Filtramos pedidos entregados con fecha y cliente validos. Recencia son dias desde la ultima compra; frecuencia cuenta pedidos; gasto suma `price` (no incluye flete); variedad cuenta categorias conocidas. `customer_unique_id` solo agrupa. Estas son aproximaciones RFM permitidas por las columnas disponibles. `log1p` modera colas derechas; `StandardScaler` iguala escalas porque distancias euclidianas son sensibles a unidades. Maryani et al. ([2018](https://doi.org/10.1109/IAC.2018.8780570)) estudian RFM con K-Means en un contexto distinto; Milligan & Cooper ([1988](https://doi.org/10.1007/BF01897163)) muestran que estandarizar afecta soluciones de clustering. Ninguna fuente prueba segmentos reales de Olist.


In [ ]:
orders = orders.merge(customers[["customer_id", "customer_unique_id"]], on="customer_id", how="inner")
delivered = orders.loc[
    orders["order_status"].eq("delivered")
    & orders["order_purchase_timestamp"].notna()
    & orders["customer_unique_id"].notna()
].copy()
if delivered.empty:
    raise ValueError("No hay pedidos entregados con fecha y cliente validos.")
reference_date = delivered["order_purchase_timestamp"].max().normalize() + pd.Timedelta(days=1)
order_map = delivered[["order_id", "customer_unique_id", "order_purchase_timestamp"]]
purchases = items.merge(order_map, on="order_id", how="inner").merge(
    products[["product_id", "product_category_name"]], on="product_id", how="left"
)
customer_features = purchases.groupby("customer_unique_id").agg(
    frequency=("order_id", "nunique"),
    spend_brl=("price", "sum"),
    category_variety=("product_category_name", "nunique"),
    last_purchase=("order_purchase_timestamp", "max"),
)
customer_features["recency_days"] = (reference_date - customer_features.pop("last_purchase")).dt.total_seconds() / 86400
feature_columns = ["recency_days", "frequency", "spend_brl", "category_variety"]
customer_features = customer_features[feature_columns].replace([np.inf, -np.inf], np.nan).dropna()
if (customer_features[feature_columns] < 0).any().any():
    raise ValueError("log1p necesita variables no negativas.")
print(f"Clientes agregados: {len(customer_features):,}; fecha de referencia: {reference_date.date()}")
display(customer_features.describe().T.round(2))


### EDA de agregados

Los histogramas en log1p muestran la forma de recencia, frecuencia, gasto y variedad; la tabla esta en unidades originales. Media, mediana y sesgo permiten ver si hay cola derecha. La correlacion ayuda a detectar variables redundantes, pero no implica una relacion causal.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 7))
for ax, column in zip(axes.flat, feature_columns):
    ax.hist(np.log1p(customer_features[column].clip(lower=0)), bins=40, color="teal", alpha=0.8)
    ax.set_title(f"{column} — log(1+x)")
    ax.set_xlabel("Valor transformado")
    ax.set_ylabel("Clientes")
plt.tight_layout()
plt.show()
summary = customer_features[feature_columns].agg(["median", "mean", "skew"]).T
summary.columns = ["mediana", "media", "sesgo"]
display(summary.round(2))
correlation = np.log1p(customer_features[feature_columns]).corr()
fig, ax = plt.subplots(figsize=(6, 5))
image = ax.imshow(correlation, vmin=-1, vmax=1, cmap="coolwarm")
ax.set_xticks(range(len(feature_columns)), feature_columns, rotation=45, ha="right")
ax.set_yticks(range(len(feature_columns)), feature_columns)
for row in range(len(feature_columns)):
    for column in range(len(feature_columns)):
        ax.text(column, row, f"{correlation.iloc[row, column]:.2f}", ha="center", va="center", fontsize=8)
ax.set_title("Correlación tras log1p — exploratoria")
fig.colorbar(image, ax=ax, label="Correlacion")
plt.tight_layout()
plt.show()


## 4. Modelado: comparar tres algoritmos

Probamos K-Means, mezcla gaussiana (GMM) y Birch con 2, 3 o 4 grupos. Son familias distintas: centroides, distribuciones gaussianas e incremental jerarquico. Se comparan con silhouette (mayor cohesion/separacion es preferible) y Davies-Bouldin (menor es preferible) en la misma muestra reproducible. Rousseeuw ([1987](https://doi.org/10.1016/0377-0427(87)90125-7)) propone silhouette como ayuda interna, no como prueba de valor comercial. Seleccionamos por silhouette y ajustamos el elegido en todos los clientes.


In [ ]:
X = customer_features[feature_columns]
selection_sample = X.sample(n=min(4000, len(X)), random_state=42)
configurations = []
for n_groups in (2, 3, 4):
    configurations.extend([
        (f"K-Means k={n_groups}", KMeans(n_clusters=n_groups, n_init=10, random_state=42)),
        (f"GMM k={n_groups}", GaussianMixture(n_components=n_groups, covariance_type="diag", reg_covar=1e-5, n_init=2, random_state=42)),
        (f"Birch k={n_groups}", Birch(n_clusters=n_groups, threshold=0.5)),
    ])
results = []
candidate_pipelines = {}
for name, estimator in configurations:
    candidate = make_pipeline(FunctionTransformer(np.log1p, feature_names_out="one-to-one"), StandardScaler(), estimator)
    candidate.fit(selection_sample)
    labels = candidate.predict(selection_sample)
    if len(np.unique(labels)) < 2:
        continue
    scaled_sample = candidate[:-1].transform(selection_sample)
    proportions = pd.Series(labels).value_counts(normalize=True)
    results.append({
        "modelo": name,
        "silhouette": silhouette_score(scaled_sample, labels),
        "Davies-Bouldin (menor mejor)": davies_bouldin_score(scaled_sample, labels),
        "grupo mas pequeno (%)": 100 * proportions.min(),
    })
    candidate_pipelines[name] = candidate
comparison = pd.DataFrame(results).sort_values(
    ["silhouette", "Davies-Bouldin (menor mejor)"], ascending=[False, True]
).reset_index(drop=True)
if comparison.empty:
    raise ValueError("Ninguna configuracion produjo al menos dos grupos.")
display(comparison.round(3))
best_name = comparison.iloc[0]["modelo"]
best_cluster_model = clone(candidate_pipelines[best_name]).fit(X)
customer_features["cluster"] = best_cluster_model.predict(X)
cluster_profile = customer_features.groupby("cluster")[feature_columns].median().round(1)
print(f"Elegido por silhouette en muestra de {len(selection_sample):,}: {best_name}. No hay etiquetas verdaderas.")
display(cluster_profile)


## 5. Evaluacion e interpretacion

PCA reduce a dos dimensiones para visualizar; pierde informacion. Revisamos el tamano de cada grupo y sus medianas en unidades originales. Un indice interno favorable no demuestra estabilidad ni utilidad. Los numeros de cluster son arbitrarios, no segmentos verdaderos.


In [ ]:
projection_sample = X.loc[selection_sample.index]
labels_sample = customer_features.loc[selection_sample.index, "cluster"]
scaled_sample = best_cluster_model[:-1].transform(projection_sample)
projection = PCA(n_components=2, random_state=42).fit_transform(scaled_sample)
plt.figure(figsize=(8, 5))
scatter = plt.scatter(projection[:, 0], projection[:, 1], c=labels_sample, cmap="tab10", alpha=0.65, s=18)
plt.xlabel("Componente principal 1")
plt.ylabel("Componente principal 2")
plt.title(f"PCA exploratoria — {best_name}")
plt.colorbar(scatter, label="Grupo asignado")
plt.tight_layout()
plt.show()
cluster_sizes = customer_features["cluster"].value_counts().sort_index().rename("clientes")
display(cluster_sizes.to_frame().join(cluster_profile))


¿Qué variables distinguen los grupos? ¿Cambia la lectura si usamos medianas en lugar de medias? Silhouette y perfiles son apoyos descriptivos, no etiquetas de negocio ni evidencia causal.

## 6. Despliegue: demo Gradio temporal

La demo devuelve el grupo del modelo elegido para agregados de ejemplo. Los numeros de grupo son etiquetas arbitrarias. La interfaz Colab es temporal, no un despliegue de produccion ni una herramienta de decisiones sobre personas.


In [ ]:
import gradio as gr

def cluster_demo(recency_days, frequency, spend_brl, category_variety):
    example = pd.DataFrame([[recency_days, frequency, spend_brl, category_variety]], columns=feature_columns)
    group = int(best_cluster_model.predict(example)[0])
    return f"Metodo: {best_name}. Grupo exploratorio {group}; no es una categoria verdadera."

demo = gr.Interface(
    fn=cluster_demo,
    inputs=[
        gr.Number(value=60, minimum=0, label="Recencia (dias)"),
        gr.Number(value=2, minimum=1, label="Pedidos observados"),
        gr.Number(value=200, minimum=0, label="Suma de precios (BRL)"),
        gr.Number(value=2, minimum=0, label="Categorias distintas"),
    ],
    outputs=gr.Textbox(label="Resultado exploratorio"),
    title="Demo de agrupamiento",
    description=f"Elegido por silhouette: {best_name}. Los grupos no son categorias verdaderas.",
)
demo.launch(share=False, inline=True, prevent_thread_lock=True)
